# **(TRANSFORM DATA PROCESS)**

## Objectives

* Clean the raw insurance dataset, manage missing values, encode categorical variables, and engineer useful features for analysis.

## Inputs

* raw_data/insurance.csv

## Outputs

* Cleaned_data.csv with cleaned, encoded, and feature-engineered data.

## Additional Comments

* This notebook prepares the dataset for exploratory analysis and predictive modelling.

---

# Change working directory

We need to make sure the project root is the active working directory before loading the dataset.

In [ ]:
import os
current_dir = os.getcwd()
if os.path.basename(current_dir) == "jupyter_notebooks":
    os.chdir(os.path.dirname(current_dir))
print(f"Current working directory: {os.getcwd()}")


# Load the raw dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("raw_data/insurance.csv")
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
df.head()


# Review data quality and missing values

In [ ]:
print(df.info())
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())


# Clean the dataset

In [ ]:
df = df.copy()

# Standardise column names and trim whitespace
df.columns = [col.strip().lower() for col in df.columns]

# Remove duplicate records
df = df.drop_duplicates().reset_index(drop=True)

# Fill missing values in a way that preserves data integrity
numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=[object]).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    mode_value = df[col].mode()
    if not mode_value.empty:
        df[col] = df[col].fillna(mode_value.iloc[0])
    else:
        df[col] = df[col].fillna("Unknown")

# Normalise string categories
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

df.head()


# Create new features

In [ ]:
# BMI categories based on standard medical thresholds
bins = [0, 18.5, 25, 30, np.inf]
labels = ["underweight", "normal", "overweight", "obese"]
df["bmi_category"] = pd.cut(df["bmi"], bins=bins, labels=labels, right=False)

# Additional feature to capture upper BMI risk
df["bmi_squared"] = df["bmi"] ** 2

df[["bmi", "bmi_category", "bmi_squared", "smoker"]].head()


# Encode categorical variables

In [ ]:
encoded_df = pd.get_dummies(df, columns=["sex", "smoker", "region", "bmi_category"], drop_first=False)
dummy_columns = [col for col in encoded_df.columns if col.startswith(("sex_", "smoker_", "region_", "bmi_category_"))]
encoded_df[dummy_columns] = encoded_df[dummy_columns].astype(int)
encoded_df.head()


# Save the cleaned dataset

In [ ]:
encoded_df.to_csv("Cleaned_data.csv", index=False)
print(f"Saved cleaned dataset with shape: {encoded_df.shape}")
print("\nColumns in cleaned dataset:")
print(encoded_df.columns.tolist())


---

* Final cleaned dataset is ready for analysis and model development.